# Лекция 14 — Паттерны проектирования: задачи для самостоятельной работы

**Темы:** паттерны GoF (порождающие, структурные, поведенческие) и их реализация на Python; ML-часть лекции: ML как конвейер, блоки-идиомы (ResNet shortcut), реестры моделей, мониторинг данных, A/B-тесты.

- **Задачи 1–10** — простые (базовые паттерны)
- **Задачи 11–20** — средние (классические GoF + pipeline)
- **Задачи 21–30** — сложные (паттерны в ML-контексте)

Все решения работают **без интернета** и сторонних библиотек (только стандартная библиотека).


## Простые задачи


### 1. Одиночка (Singleton)
Реализуйте класс `Singleton` так, чтобы все попытки создать объект давали один и тот же экземпляр (переопределите `__new__`). Проверьте: `a is b`, а значение, записанное через `a`, видно через `b`.

**Подсказка:** храните единственный экземпляр в атрибуте класса `_instance` и создавайте его только если он `None`.


### 2. Фабричный метод (Factory Method)
Создайте иерархию `Shape` → `Circle`, `Square` с методом `area()` и фабричную функцию `shape_factory(kind, size)`, возвращающую объект по строке `'circle'`/`'square'`. Неизвестный тип должен вызывать `KeyError`.

**Подсказка:** словарь `{имя: класс}` внутри фабрики вместо ветвления `if/elif`.


### 3. Строитель (Builder)
Реализуйте `QueryBuilder('users')` с цепочкой методов `.select(['a','b'])`, `.where("age > 18")`, `.limit(10)` и `.build()`, возвращающим SQL-строку:

```sql
SELECT a, b FROM users WHERE age > 18 LIMIT 10
```

Каждый метод возвращает `self` — это «флюент-API». Без вызовов `build()` должен давать `SELECT * FROM users`.

**Подсказка:** метод `build()` собирает строку из накопленных частей; пропущенные секции просто не добавляются.


### 4. Прототип (Prototype)
Класс `Config(name, params)` содержит список параметров. Реализуйте метод `clone()` через `copy.deepcopy` и покажите разницу с `copy.copy`: изменение вложенного словаря после глубокого копирования не трогает оригинал, а после мелкого — трогает.

**Подсказка:** `copy.copy` копирует только верхний объект, вложенные списки/словари остаются общими.


### 5. Адаптер (Adapter)
Есть «наследованный» `LegacyExcel` с методом `read_excel_data()` (список списков). Напишите `ExcelAdapter` с современным интерфейсом `.fetch()`, возвращающим список словарей.

```python
LegacyExcel().read_excel_data()  # [['a', 1], ['b', 2]]
ExcelAdapter(...).fetch()        # [{'col1': 'a', 'col2': 1}, {'col1': 'b', 'col2': 2}]
```

**Подсказка:** адаптер хранит объект старого класса внутри себя и транслирует один интерфейс в другой.


### 6. Декоратор Python: счётчик вызовов
Напишите декоратор `@counter`, который увеличивает атрибут `func.calls` при каждом вызове функции. Сохраните имя функции (`functools.wraps`). Проверьте: после трёх вызовов `add.calls == 3`, `add.__name__ == 'add'`.

**Подсказка:** атрибут `calls` повесьте на саму обёртку до возврата: `wrapper.calls = 0`.


### 7. Фасад (Facade)
Класс `CsvFacade(path)` с одним методом `.load()`: открывает файл, разбирает через `csv.DictReader` и возвращает список словарей с числами, отбрасывая полностью пустые строки. Функция-фасад скрывает три шага (открыть→разобрать→очистить) за одним вызовом.

**Подсказка:** `csv.DictReader(f)` даёт словари со строковыми значениями; приведите `float(v)`, а строку, где все значения `''`, пропустите.


### 8. Стратегия (Strategy)
Корзина `Cart(price)` с атрибутом-стратегией `discount` — функцией `цена → итог`: без скидки, `10%`, `20% если цена > 1000`. Покажите, что смена функции меняет поведение без изменения класса.

```python
cart = Cart(500); cart.discount = ten_percent
```

**Подсказка:** стратегии — обычные функции с одним signature `(price) -> float`; класс хранит ссылку на функцию.


### 9. Наблюдатель (Observer)
Реализуйте `EventBus` с `subscribe(event, fn)` и `notify(event, data)`. Подпишите две функции на событие `'epoch'` и убедитесь, что `notify` вызывает обе в порядке подписки (тренировка логгера и графика обучения).

**Подсказка:** словарь `событие → список коллбэков`; `notify` просто обходит список.


### 10. Итератор (Iterator)
Реализуйте `FibIterator(limit)` — протокол `__iter__`/`__next__`, выдающий числа Фибоначчи до `limit` включительно. Проверьте: `list(FibIterator(34))` даёт 10 элементов, `sum(FibIterator(10)) == 20`.

**Подсказка:** в `__iter__` сбросьте состояние и верните `self`; в `__next__` поднимите `StopIteration` при выходе за limit.


## Средние задачи


### 11. Абстрактная фабрика (Abstract Factory)
Семейства виджетов `LightUI` и `DarkUI` создают согласованные пары: кнопка + поле ввода в одном стиле. Функция `render_ui(factory)` не знает, какая тема конкретная. Проверьте оба семейства.

**Подсказка:** фабрика — класс с методами `create_button()` и `create_input()`; интерфейс фиксирует набор, реализации — темы.


### 12. Компоновщик (Composite)
Дерево арифметических выражений: `Number(2)`, `Add(left, right)`, `Mul(left, right)` — все с методом `eval()`. Постройте `(2 + 3) * 4` и получите 20. Листья и ветки используют один интерфейс.

**Подсказка:** у составного узла `eval()` = операция над `eval()` детей; рекурсия завершается на `Number`.


### 13. Заместитель (Proxy): ленивая загрузка
`LazyProxy` не создаёт «тяжёлый» датасет, пока к нему не обратятся, и создаёт его ровно один раз (счётчик `loads`). Проверьте: после создания прокси `loads == 0`, после двух обращений к `.rows` — `loads == 1`, обе выдачи — один и тот же объект.

**Подсказка:** реализуйте ленивое свойство: при `self._real is None` создайте объект и увеличьте счётчик.


### 14. Шаблонный метод (Template Method)
Базовый класс `ReportPipeline` с фиксированным `run()`: `load → clean → render`, где `clean` (отбросить `None`) реализован в базе, а `load`/`render` — абстрактные хуки. Реализуйте `SumReport` и `AvgReport`. Проверьте: `SumReport([1, None, 3]).run() == 'sum=4'`.

**Подсказка:** инверсия управления: родительский метод вызывает шаги, которые переопределяет потомок.


### 15. Команда (Command) с отменой
Классы `TurnOn`/`TurnOff` (light) с `execute()`/`undo()`, пульт `Remote` с историей и кнопкой `undo()`. Последовательность on → off → undo оставляет свет включённым.

**Подсказка:** команда хранит получателя; `undo` делает операцию, противоположную `execute`.


### 16. Цепочка ответственности (Chain of Responsibility)
Валидация имени: `NotEmpty` → `MaxLen(10)` → `NoDigits`, каждый обработчик вызывает следующего и собирает ошибки. Проверьте: `'hello'` — чисто, `'user123'` — одна ошибка, `''` — минимум одна.

**Подсказка:** у обработчика есть ссылка `next`; базовый `handle` вызывает свой `check` и передаёт эстафету.


### 17. Состояние (State)
Турникет: состояния `LockedState`/`UnlockedState` — классы с методами `coin()` и `push()`. Монета разблокирует, проход разблокированного считает проход и блокирует; монета в unlocked ничего не меняет. Проверьте сценарий из лекции.

**Подсказка:** поведение объекта определяется текущим объектом-состоянием; методы-обработчики возвращают новое состояние.


### 18. Приспособленец (Flyweight)
Рендер текста: объекты-глифы создаются только один раз на уникальный символ (пул-кэш), список гифов для строки `'абракадабра'` содержит 11 ссылок при 5 созданных объектах; повторы — один и тот же объект (`is`).

**Подсказка:** словарь-пул `{символ: глиф}`; счётчик `created` на классе.


### 19. Мост (Bridge)
Абстракция `RemoteButton.toggle()` работает с любой реализацией `Device` (`Lamp`, `Speaker` — у каждого своё состояние). Один и тот же код пульта переключает разные устройства. Покажите это в цикле.

**Подсказка:** «ручка» и «реализация» разделены: абстракция хранит ссылку на объект-реализацию, а не наследует её.


### 20. ML как конвейер (Pipeline)
Реализуйте `Pipeline(steps)` в духе sklearn: у шага есть `fit(data)` (опционально) и `transform(data)`, данные последовательно протекают через шаги. Соберите `[Standardize(), Clip(-1.5, 1.5)]` и прогоните `[10, 12, 11, 9, 1000]`. После клипа максимум равен 1.5 — выброс обезврежен, а среднее близко к 0.

**Подсказка:** `Standardize.fit` запоминает среднее и std, `Clip` — только `transform`; шаг-функция не обязана иметь `fit`.


## Сложные задачи


### 21. Реестр слоёв: декоратор + фабрика
Как в Keras: реестр `LAYERS` и декоратор `@register('relu')`, регистрирующий класс. Реализуйте слои `ReLU` и `Sigmoid` (с `math.exp`) с методом `forward(xs)`. Соберите слой по имени из реестра: `LAYERS['sigmoid']().forward([0]) == [0.5]`.

**Подсказка:** декоратор с параметром — это функция, возвращающая декоратор; внутри `deco(cls)` кладёт класс в словарь и возвращает `cls`.


### 22. Наблюдатель для мониторинга данных (data drift)
ML-система следит за признаками: `DriftMonitor.check(name, baseline, current)` сравнивает средние и при сдвиге больше порога добавляет алерт и возвращает `True`. Сгенерируйте baseline `N(0,1)` и «прод» `N(3,1)` (1000 точек, seed) — алерт сработал; на второй чистой фиче — нет.

**Подсказка:** простейшая статистика drift — разница средних (или std); подписчики получают `(имя, величина)`.


### 23. Строитель модели (Sequential-конфиг)
`ModelBuilder().add_dense(64).add_dense(10).build()` возвращает список блоков. Функция `count_params(model, in_dim)` считает веса по формуле Dense: `in*out + bias`, передавая размерность дальше. Для входа 784: 784·64 + 64 + 64·10 + 10.

**Подсказка:** builder накапливает блоки и возвращает `self`; расчёт — чистая функция над конфигом.


### 24. Остаточный блок (ResNet-идиома)
`ResidualBlock(inner)` вычисляет `inner(x) + x` — shortcut. Проверьте: при `inner = lambda x: 0` блок работает как тождественность (`block(7) == 7`), при `inner = lambda x: 2x` для списка — удваивает сумму с оригиналом. Почему shortcut спасает глубокие сети от затухания градиента?

**Подсказка:** градиент проходит через «магистраль» `+x` даже при обученном в ноль inner-блоке.


### 25. kNN со стратегией расстояния
Реализуйте `KNN(k, distance=euclidean)`: `fit(X, y)`, `predict(x)` — голосование большинства среди k ближайших. Стратегии расстояний: `euclidean`, `manhattan`. На двух облаках точек классифицируйте новые точки обоими способами.

**Подсказка:** отсортируйте пары `(расстояние, метка)` и посчитайте `Counter` по первым k меткам.


### 26. Прокси с кэшированием (Memoize)
`CachingProxy` над дорогим `SlowService.compute(x)`: 100 одинаковых запросов → один реальный вызов (`svc.calls == 1`), разные аргументы кэшируются отдельно. Сравните с `functools.lru_cache`.

**Подсказка:** словарь-кэш по аргументу; посчитайте, сколько раз реально дошёл вызов до сервиса.


### 27. Синглтон на метаклассе (и MRO)
`SingletonMeta(type)` с `_instances` по типам: `Config()` — один экземпляр, а подкласс `ProdConfig` получает свой собственный (словарь по классам, не общий!). Проверьте `a is b`, `p is not a`.

**Подсказка:** переопределите `__call__` метакласса и кешируйте `cls(...)`, ключ — сам класс `cls`.


### 28. Grid search по конвейеру
На синтетических двухклассовых данных (60 точек, центры (0,0) и (3.2, 3.2) с лёгким перекрытием, `seed=7`) переберите `k ∈ {1,3,5}` × `{euclidean, manhattan}` с проверкой leave-one-out и выберите конфигурацию с лучшей точностью. Точность лучшей конфигурации — выше 0.9.

**Подсказка:** leave-one-out: для каждой точки предсказание по остальным; используйте `itertools.product` и `KNN` из задачи 25.


### 29. A/B-тест и shadow-режим
Симуляция: модель A (качество 0.80) обслуживает 90% трафика, модель B — 10% (canary), плюс shadow — предсказание B «вхолостую» на весь трафик. За 10 000 запросов с фиксированным seed: доля canary-трафика ≈ 10%, точность B выше A.

**Подсказка:** `random.Random(0)` для детерминизма; считайте хиты и успехи отдельно для A и B.


### 30. Мини-ML-система: три паттерна сразу
Дан маленький датасет выживания (CSV-строка: класс, пол, возраст, выжил). Стратегия-правило `'women_children'` (`female` или `age < 12`) против правила `'first_class'`. Observer `MetricsLog` собирает точность каждого правила. На этой выборке первое правило даёт 1.0, второе — 0.5. Сделайте фасад-функцию `evaluate(rules)`, возвращающую имя лучшего правила.

**Подсказка:** правила — функции-стратегии `row -> предсказание`; точность доля совпадений; логгер — подписчик на результат прогона.


## Где потренироваться дальше

- [Refactoring.Guru — Паттерны проектирования](https://refactoring.guru/ru/design-patterns)
- [SourceMaking — Design Patterns](https://sourcemaking.com/design_patterns)
- [Паттерны проектирования на Python](https://southbridge-institute.com/tpost/424o5b42v1-patterny-proektovaniya-kod-na-python)
- [Ч. Хуен. Designing Machine Learning Systems](https://www.oreilly.com/library/view/designing-machine-learning/9781098107956/)
- [Habr: паттерны в машинном обучении](https://habr.com/ru/articles/438802/)
